In [4]:

import requests
import time
import datetime
from openpyxl import Workbook
from openpyxl.styles import (
    Font, PatternFill, Alignment, Border, Side, GradientFill
)
from openpyxl.utils import get_column_letter


QUERY        = "Semantic SLAM"
YEAR_START   = 2020
YEAR_END     = 2026
FETCH_POOL   = 200       # candidates fetched before scoring
TOP_N        = 50
OUTPUT_FILE  = "semantic_slam_top50.xlsx"

# Scoring weights (must sum to 1.0)
W_CITATIONS      = 0.40
W_VENUE          = 0.25
W_RECENCY        = 0.15
W_CITATION_RATE  = 0.20

VENUE_TIERS = {
    "cvpr": 1.0, "iccv": 1.0, "eccv": 1.0, "nips": 1.0, "neurips": 1.0,
    "iclr": 1.0, "icml": 1.0, "rss": 1.0,
    "icra": 0.85, "iros": 0.85, "aaai": 0.85, "ijcai": 0.85,
    "3dv": 0.85, "ismar": 0.85, "corl": 0.85,
    "t-ro": 0.75, "ral": 0.75, "ra-l": 0.75, "ijrr": 0.75,
    "tpami": 0.75, "ijcv": 0.75, "sensors": 0.65,
    "arxiv": 0.40,
}

def venue_score(venue_str: str) -> float:
    if not venue_str:
        return 0.30
    v = venue_str.lower()
    for key, score in VENUE_TIERS.items():
        if key in v:
            return score
    return 0.50

# FETCH FROM SEMANTIC SCHOLAR

def fetch_papers(query: str, year_start: int, year_end: int, limit: int) -> list:
    """
    Calls the free Semantic Scholar Graph API.
    No API key required for basic usage (100 req/5 min limit).
    Docs: https://api.semanticscholar.org/api-docs/
    """
    api_url = "https://api.semanticscholar.org/graph/v1/paper/search"
    fields  = (
        "title,authors,year,citationCount,venue,"
        "externalIds,abstract,publicationTypes,openAccessPdf"
    )

    papers = []
    offset = 0
    batch  = 100

    print(f"\nFetching '{query}' papers ({year_start}–{year_end}) ...")

    while len(papers) < limit:
        params = {
            "query":  query,
            "fields": fields,
            "limit":  min(batch, limit - len(papers)),
            "offset": offset,
            "year":   f"{year_start}-{year_end}",
        }

        resp = requests.get(api_url, params=params, timeout=30)

        if resp.status_code == 429:
            print("  Rate limited — waiting 15 s ...")
            time.sleep(15)
            continue

        if resp.status_code != 200:
            print(f"  API error {resp.status_code}: {resp.text[:300]}")
            break

        data         = resp.json()
        batch_papers = data.get("data", [])
        if not batch_papers:
            break

        papers.extend(batch_papers)
        offset += len(batch_papers)
        total   = data.get("total", "?")
        print(f"  Fetched {len(papers)} / {min(limit, total if isinstance(total, int) else limit)}")
        time.sleep(1)   # polite rate-limiting

    return papers

def score_paper(paper: dict, max_citations: int, max_rate: float, current_year: int) -> float:
    citations = paper.get("citationCount") or 0
    year      = paper.get("year") or current_year
    venue     = paper.get("venue") or ""
    age       = max(current_year - year, 1)

    cite_norm = citations / max_citations if max_citations > 0 else 0
    rate      = citations / age
    rate_norm = rate / max_rate if max_rate > 0 else 0
    recency   = 1.0 - min((current_year - year) / 10, 1.0) * 0.5
    venue_s   = venue_score(venue)

    raw = (
        W_CITATIONS     * cite_norm +
        W_VENUE         * venue_s   +
        W_RECENCY       * recency   +
        W_CITATION_RATE * rate_norm
    )
    return round(raw * 100, 1)

def rank_papers(papers: list, top_n: int, current_year: int) -> list:
    papers = [p for p in papers if p.get("title") and p.get("year")]

    seen, unique = set(), []
    for p in papers:
        key = p["title"].lower().strip()
        if key not in seen:
            seen.add(key)
            unique.append(p)

    if not unique:
        return []

    max_citations = max((p.get("citationCount") or 0) for p in unique)
    max_rate      = max(
        (p.get("citationCount") or 0) / max(current_year - (p.get("year") or current_year), 1)
        for p in unique
    )

    for p in unique:
        p["_score"] = score_paper(p, max_citations, max_rate, current_year)

    return sorted(unique, key=lambda x: x["_score"], reverse=True)[:top_n]

def build_url(paper: dict) -> str:
    """
    Priority: open-access PDF → ArXiv → DOI → Semantic Scholar page
    Returns the best available URL for the paper.
    """
    # 1. Open-access PDF (best — direct full text)
    pdf_url = (paper.get("openAccessPdf") or {}).get("url", "")
    if pdf_url:
        return pdf_url

    ids = paper.get("externalIds") or {}

    # 2. ArXiv abstract page
    arxiv = ids.get("ArXiv")
    if arxiv:
        return f"https://arxiv.org/abs/{arxiv}"

    # 3. DOI resolver
    doi = ids.get("DOI")
    if doi:
        return f"https://doi.org/{doi}"

    # 4. Semantic Scholar paper page (always available)
    paper_id = paper.get("paperId", "")
    if paper_id:
        return f"https://www.semanticscholar.org/paper/{paper_id}"

    return ""

# ── FORMAT ROW

def format_paper(rank: int, p: dict) -> dict:
    authors    = p.get("authors") or []
    author_str = (
        authors[0]["name"] + (" et al." if len(authors) > 1 else "")
        if authors else "Unknown"
    )

    pub_types = p.get("publicationTypes") or []
    is_survey = any(t.lower() in ["review", "survey"] for t in pub_types)

    return {
        "rank":      rank,
        "type":      "Survey" if is_survey else "Research",
        "title":     p.get("title", ""),
        "authors":   author_str,
        "year":      p.get("year"),
        "venue":     p.get("venue", ""),
        "citations": p.get("citationCount", 0),
        "score":     p["_score"],
        "abstract":  (p.get("abstract") or "")[:500],
        "url":       build_url(p),
    }

# ── EXCEL EXPORT

# Colour palette
C_HEADER_BG   = "2E3A59"   # dark navy
C_HEADER_FG   = "FFFFFF"
C_TOP10_BG    = "EDE9FC"   # soft purple
C_SURVEY_BG   = "E6F5EE"   # soft teal
C_ALT_BG      = "F7F8FA"   # light grey stripe
C_WHITE       = "FFFFFF"
C_SCORE_HIGH  = "1D9E75"   # green
C_SCORE_MID   = "E6900A"   # amber
C_SCORE_LOW   = "A32D2D"   # red
C_BORDER      = "D0D5E0"

def thin_border():
    s = Side(style="thin", color=C_BORDER)
    return Border(left=s, right=s, top=s, bottom=s)

def header_fill():
    return PatternFill("solid", fgColor=C_HEADER_BG)

def score_color(score: float) -> str:
    if score >= 75:
        return C_SCORE_HIGH
    elif score >= 50:
        return C_SCORE_MID
    return C_SCORE_LOW

def save_excel(papers: list, filepath: str):
    wb = Workbook()

    # ── Sheet 1: Full ranked list
    ws = wb.active
    ws.title = "Top 50 Papers"

    headers = [
        "Rank", "Type", "Title", "Authors", "Year",
        "Venue", "Citations", "Score", "Abstract", "URL"
    ]
    col_widths = [6, 10, 60, 28, 6, 22, 11, 8, 80, 50]

    # Header row
    for col, (h, w) in enumerate(zip(headers, col_widths), start=1):
        cell = ws.cell(row=1, column=col, value=h)
        cell.font      = Font(bold=True, color=C_HEADER_FG, name="Arial", size=11)
        cell.fill      = header_fill()
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border    = thin_border()
        ws.column_dimensions[get_column_letter(col)].width = w

    ws.row_dimensions[1].height = 28
    ws.freeze_panes = "A2"

    # Data rows
    for row_idx, p in enumerate(papers, start=2):
        is_top10  = p["rank"] <= 10
        is_survey = p["type"] == "Survey"
        is_alt    = row_idx % 2 == 0

        if is_top10:
            row_bg = C_TOP10_BG
        elif is_survey:
            row_bg = C_SURVEY_BG
        elif is_alt:
            row_bg = C_ALT_BG
        else:
            row_bg = C_WHITE

        row_fill = PatternFill("solid", fgColor=row_bg)

        values = [
            p["rank"], p["type"], p["title"], p["authors"],
            p["year"], p["venue"], p["citations"], p["score"],
            p["abstract"], p["url"]
        ]

        for col, val in enumerate(values, start=1):
            cell        = ws.cell(row=row_idx, column=col, value=val)
            cell.fill   = row_fill
            cell.border = thin_border()
            cell.font   = Font(name="Arial", size=10)
            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True,
                horizontal="center" if col in (1, 5, 7, 8) else "left"
            )

        # Score cell: coloured font
        score_cell      = ws.cell(row=row_idx, column=8)
        score_cell.font = Font(
            name="Arial", size=10, bold=True,
            color=score_color(p["score"])
        )

        # URL cell: hyperlink
        if p["url"]:
            url_cell = ws.cell(row=row_idx, column=10)
            url_cell.hyperlink = p["url"]
            url_cell.value     = p["url"]
            url_cell.font      = Font(
                name="Arial", size=10, color="185FA5",
                underline="single"
            )

        # Row height based on abstract length
        ws.row_dimensions[row_idx].height = min(max(len(p["abstract"]) / 6, 18), 120)

    # Auto-filter
    ws.auto_filter.ref = f"A1:{get_column_letter(len(headers))}1"

    # ── Sheet 2: Top 10 summary ───────────────────────────────────────────────
    ws2       = wb.create_sheet("Top 10 Summary")
    top10     = [p for p in papers if p["rank"] <= 10]
    sum_hdrs  = ["Rank", "Title", "Authors", "Year", "Venue", "Citations", "Score", "URL"]
    sum_widths = [6, 65, 28, 6, 24, 11, 8, 55]

    for col, (h, w) in enumerate(zip(sum_hdrs, sum_widths), start=1):
        cell            = ws2.cell(row=1, column=col, value=h)
        cell.font       = Font(bold=True, color=C_HEADER_FG, name="Arial", size=11)
        cell.fill       = header_fill()
        cell.alignment  = Alignment(horizontal="center", vertical="center")
        cell.border     = thin_border()
        ws2.column_dimensions[get_column_letter(col)].width = w

    ws2.row_dimensions[1].height = 28
    ws2.freeze_panes = "A2"

    for row_idx, p in enumerate(top10, start=2):
        row_bg   = C_TOP10_BG if row_idx % 2 == 0 else C_WHITE
        row_fill = PatternFill("solid", fgColor=row_bg)
        vals = [p["rank"], p["title"], p["authors"], p["year"],
                p["venue"], p["citations"], p["score"], p["url"]]
        for col, val in enumerate(vals, start=1):
            cell        = ws2.cell(row=row_idx, column=col, value=val)
            cell.fill   = row_fill
            cell.border = thin_border()
            cell.font   = Font(name="Arial", size=10)
            cell.alignment = Alignment(
                vertical="center", wrap_text=True,
                horizontal="center" if col in (1, 4, 6, 7) else "left"
            )
        score_cell      = ws2.cell(row=row_idx, column=7)
        score_cell.font = Font(name="Arial", size=10, bold=True,
                               color=score_color(p["score"]))
        if p["url"]:
            url_cell           = ws2.cell(row=row_idx, column=8)
            url_cell.hyperlink = p["url"]
            url_cell.font      = Font(name="Arial", size=10, color="185FA5",
                                      underline="single")
        ws2.row_dimensions[row_idx].height = 30

    ws2.auto_filter.ref = f"A1:{get_column_letter(len(sum_hdrs))}1"

    # ── Sheet 3: Surveys only
    ws3      = wb.create_sheet("Survey Papers")
    surveys  = [p for p in papers if p["type"] == "Survey"]

    for col, (h, w) in enumerate(zip(sum_hdrs, sum_widths), start=1):
        cell           = ws3.cell(row=1, column=col, value=h)
        cell.font      = Font(bold=True, color=C_HEADER_FG, name="Arial", size=11)
        cell.fill      = PatternFill("solid", fgColor="0F6E56")
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border    = thin_border()
        ws3.column_dimensions[get_column_letter(col)].width = w

    ws3.row_dimensions[1].height = 28
    ws3.freeze_panes = "A2"

    for row_idx, p in enumerate(surveys, start=2):
        row_fill = PatternFill("solid", fgColor=C_SURVEY_BG if row_idx % 2 == 0 else C_WHITE)
        vals = [p["rank"], p["title"], p["authors"], p["year"],
                p["venue"], p["citations"], p["score"], p["url"]]
        for col, val in enumerate(vals, start=1):
            cell        = ws3.cell(row=row_idx, column=col, value=val)
            cell.fill   = row_fill
            cell.border = thin_border()
            cell.font   = Font(name="Arial", size=10)
            cell.alignment = Alignment(
                vertical="center", wrap_text=True,
                horizontal="center" if col in (1, 4, 6, 7) else "left"
            )
        if p["url"]:
            url_cell           = ws3.cell(row=row_idx, column=8)
            url_cell.hyperlink = p["url"]
            url_cell.font      = Font(name="Arial", size=10, color="185FA5",
                                      underline="single")
        ws3.row_dimensions[row_idx].height = 30

    # ── Sheet 4: Scoring legend
    ws4 = wb.create_sheet("Scoring Method")
    ws4.column_dimensions["A"].width = 28
    ws4.column_dimensions["B"].width = 55

    legend_data = [
        ("SCORING FORMULA", ""),
        ("Factor", "Weight"),
        ("Citation count (normalised)", "40% — direct community impact signal"),
        ("Citation rate (citations/year)", "20% — corrects bias against recent papers"),
        ("Venue prestige", "25% — CVPR/RSS > preprint; filters low-quality work"),
        ("Recency bonus", "15% — slight boost so 2024 papers aren't buried"),
        ("", ""),
        ("VENUE TIERS", ""),
        ("Tier 1 (score 1.0)", "CVPR, ICCV, ECCV, NeurIPS, ICLR, ICML, RSS"),
        ("Tier 2 (score 0.85)", "ICRA, IROS, AAAI, IJCAI, 3DV, ISMAR, CoRL"),
        ("Tier 3 (score 0.75)", "T-RO, RA-L, IJRR, TPAMI, IJCV"),
        ("Tier 4 (score 0.65)", "Sensors, other journals"),
        ("Preprint (score 0.40)", "arXiv"),
        ("", ""),
        ("COLOUR CODING", ""),
        ("Purple row", "Top 10 overall best papers"),
        ("Teal row",   "Survey / review paper"),
        ("Score ≥ 75", "Green — high impact"),
        ("Score 50–74","Amber — moderate impact"),
        ("Score < 50", "Red — lower impact"),
        ("", ""),
        ("DATA SOURCE", ""),
        ("API", "Semantic Scholar Graph API (free, no key required)"),
        ("API URL", "https://api.semanticscholar.org/graph/v1/paper/search"),
        ("Paper URL priority", "1) Open-access PDF  2) ArXiv  3) DOI  4) Semantic Scholar page"),
        ("Generated", datetime.datetime.now().strftime("%Y-%m-%d %H:%M")),
    ]

    for row_idx, (a, b) in enumerate(legend_data, start=1):
        ca = ws4.cell(row=row_idx, column=1, value=a)
        cb = ws4.cell(row=row_idx, column=2, value=b)
        if a in ("SCORING FORMULA", "VENUE TIERS", "COLOUR CODING", "DATA SOURCE"):
            for c in (ca, cb):
                c.font = Font(bold=True, color=C_HEADER_FG, name="Arial", size=11)
                c.fill = header_fill()
        elif a == "Factor":
            for c in (ca, cb):
                c.font = Font(bold=True, name="Arial", size=10)
        else:
            for c in (ca, cb):
                c.font = Font(name="Arial", size=10)
        ws4.row_dimensions[row_idx].height = 18

    wb.save(filepath)
    print(f"\nSaved → {filepath}")

# ── MAIN

def main():
    current_year = datetime.datetime.now().year

    raw    = fetch_papers(QUERY, YEAR_START, YEAR_END, FETCH_POOL)
    ranked = rank_papers(raw, TOP_N, current_year)

    if not ranked:
        print("No papers found. Check your network connection or API status.")
        return

    results = [format_paper(i + 1, p) for i, p in enumerate(ranked)]
    save_excel(results, OUTPUT_FILE)

    print("\nTop 10 preview:")
    print(f"{'#':>3}  {'Score':>6}  {'Year'}  {'Title':<60}  URL")
    print("-" * 100)
    for p in results[:10]:
        url_preview = p["url"][:45] + "..." if len(p["url"]) > 45 else p["url"]
        print(f"{p['rank']:>3}  {p['score']:>6.1f}  {p['year']}  {p['title'][:60]:<60}  {url_preview}")

if __name__ == "__main__":
    main()


Fetching 'Semantic SLAM' papers (2020–2026) ...
  Rate limited — waiting 15 s ...
  Rate limited — waiting 15 s ...
  Rate limited — waiting 15 s ...
  Rate limited — waiting 15 s ...
  Fetched 100 / 200
  Rate limited — waiting 15 s ...
  Rate limited — waiting 15 s ...
  Fetched 200 / 200

Saved → semantic_slam_top50.xlsx

Top 10 preview:
  #   Score  Year  Title                                                         URL
----------------------------------------------------------------------------------------------------
  1    78.7  2022  YOLO-SLAM: A semantic SLAM system towards dynamic environmen  https://doi.org/10.1007/s00521-021-06764-3
  2    77.9  2021  Kimera-Multi: Robust, Distributed, Dense Metric-Semantic SLA  https://ieeexplore.ieee.org/ielx7/8860/985159...
  3    68.6  2024  SGS-SLAM: Semantic Gaussian Splatting For Neural Dense SLAM   https://arxiv.org/abs/2402.03246
  4    48.3  2024  A Survey of Visual SLAM in Dynamic Environment: The Evolutio  https://doi.org/10.11

# New Section